# Pergunta 3 — Qual item de infraestrutura tem maior correlação com o IDEB?

Análise da relação entre infraestrutura escolar e desempenho no IDEB
nos municípios de Alagoas, usando dados do Censo Escolar 2023 e IDEB 2023.

**Inputs:**
- `data/raw/inep/microdados_censo_escolar_2023/dados/microdados_ed_basica_2023.csv`
- `data/processed/ideb_series_al.parquet`

In [2]:
import pandas as pd
from pathlib import Path
import sys

sys.path.insert(0, str(Path().resolve().parent))
from src.config import DATA_INEP, IDEB_SERIES_PARQUET

censo_path = (
    DATA_INEP
    / "microdados_censo_escolar_2023"
    / "dados"
    / "microdados_ed_basica_2023.csv"
)

# Ler apenas a primeira linha para ver os nomes das colunas
cols = pd.read_csv(
    censo_path,
    sep=";",
    encoding="latin-1",
    nrows=1,
).columns.tolist()

print(f"Total de colunas: {len(cols)}")
print("\nColunas com prefixo IN_ (indicadores de infraestrutura):")
for c in cols:
    if c.startswith("IN_"):
        print(f"  {c}")

Total de colunas: 408

Colunas com prefixo IN_ (indicadores de infraestrutura):
  IN_VINCULO_SECRETARIA_EDUCACAO
  IN_VINCULO_SEGURANCA_PUBLICA
  IN_VINCULO_SECRETARIA_SAUDE
  IN_VINCULO_OUTRO_ORGAO
  IN_PODER_PUBLICO_PARCERIA
  IN_FORMA_CONT_TERMO_COLABORA
  IN_FORMA_CONT_TERMO_FOMENTO
  IN_FORMA_CONT_ACORDO_COOP
  IN_FORMA_CONT_PRESTACAO_SERV
  IN_FORMA_CONT_COOP_TEC_FIN
  IN_FORMA_CONT_CONSORCIO_PUB
  IN_FORMA_CONT_MU_TERMO_COLAB
  IN_FORMA_CONT_MU_TERMO_FOMENTO
  IN_FORMA_CONT_MU_ACORDO_COOP
  IN_FORMA_CONT_MU_PREST_SERV
  IN_FORMA_CONT_MU_COOP_TEC_FIN
  IN_FORMA_CONT_MU_CONSORCIO_PUB
  IN_FORMA_CONT_ES_TERMO_COLAB
  IN_FORMA_CONT_ES_TERMO_FOMENTO
  IN_FORMA_CONT_ES_ACORDO_COOP
  IN_FORMA_CONT_ES_PREST_SERV
  IN_FORMA_CONT_ES_COOP_TEC_FIN
  IN_FORMA_CONT_ES_CONSORCIO_PUB
  IN_MANT_ESCOLA_PRIVADA_EMP
  IN_MANT_ESCOLA_PRIVADA_ONG
  IN_MANT_ESCOLA_PRIVADA_OSCIP
  IN_MANT_ESCOLA_PRIV_ONG_OSCIP
  IN_MANT_ESCOLA_PRIVADA_SIND
  IN_MANT_ESCOLA_PRIVADA_SIST_S
  IN_MANT_ESCOLA_PRIVADA_S_FINS

In [3]:
# Colunas de infraestrutura selecionadas para análise
COLUNAS_INFRA = [
    "CO_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "TP_DEPENDENCIA",        # 1=Federal, 2=Estadual, 3=Municipal, 4=Privada
    "TP_SITUACAO_FUNCIONAMENTO",  # 1=Em atividade
    "IN_BIBLIOTECA",
    "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_LABORATORIO_CIENCIAS",
    "IN_QUADRA_ESPORTES",
    "IN_INTERNET",
    "IN_COMPUTADOR",
    "IN_BANDA_LARGA",
]

# Carregar apenas AL (CO_UF = 27) e escolas em atividade
# Usando chunksize para não carregar 210MB de uma vez
chunks = []
for chunk in pd.read_csv(
    censo_path,
    sep=";",
    encoding="latin-1",
    usecols=COLUNAS_INFRA,
    chunksize=10_000,
):
    filtrado = chunk[
        (chunk["CO_UF"] == 27) &
        (chunk["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (chunk["TP_DEPENDENCIA"].isin([2, 3]))  # apenas público (estadual + municipal)
    ]
    if not filtrado.empty:
        chunks.append(filtrado)

df_censo = pd.concat(chunks, ignore_index=True)

print(f"Escolas públicas em atividade em AL: {len(df_censo)}")
print(f"Municípios únicos: {df_censo['CO_MUNICIPIO'].nunique()}")
print(f"\nAmostra:")
df_censo.head(3)

Escolas públicas em atividade em AL: 2353
Municípios únicos: 102

Amostra:


,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,TP_DEPENDENCIA,TP_SITUACAO_FUNCIONAMENTO,IN_BIBLIOTECA,IN_BIBLIOTECA_SALA_LEITURA,IN_LABORATORIO_CIENCIAS,IN_LABORATORIO_INFORMATICA,IN_QUADRA_ESPORTES,IN_COMPUTADOR,IN_INTERNET,IN_BANDA_LARGA
0,27,Água Branca,2700102,2,1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,27,Água Branca,2700102,3,1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,27,Água Branca,2700102,3,1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0


In [4]:
# Agregar de escola para município
# Percentual de escolas com cada item de infraestrutura
ITENS_INFRA = [
    "IN_BIBLIOTECA",
    "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_LABORATORIO_CIENCIAS",
    "IN_QUADRA_ESPORTES",
    "IN_INTERNET",
    "IN_COMPUTADOR",
    "IN_BANDA_LARGA",
]

df_infra_mun = (
    df_censo
    .groupby("CO_MUNICIPIO")[ITENS_INFRA]
    .mean()        # média de 0/1 = percentual de escolas com o item
    .multiply(100) # converter para percentual
    .round(1)
    .reset_index()
)

# Converter CO_MUNICIPIO para string para fazer o merge com o IDEB
df_infra_mun["CO_MUNICIPIO"] = df_infra_mun["CO_MUNICIPIO"].astype(str)

print(f"Municípios no dataset de infraestrutura: {len(df_infra_mun)}")
print(f"\nEstatísticas descritivas:")
df_infra_mun[ITENS_INFRA].describe().round(1)

Municípios no dataset de infraestrutura: 102

Estatísticas descritivas:


,IN_BIBLIOTECA,IN_BIBLIOTECA_SALA_LEITURA,IN_LABORATORIO_INFORMATICA,IN_LABORATORIO_CIENCIAS,IN_QUADRA_ESPORTES,IN_INTERNET,IN_COMPUTADOR,IN_BANDA_LARGA
count,102.0,102.0,102.0,102.0,102.0,102.0,102.0,102.0
mean,17.5,34.1,17.0,8.5,19.5,91.5,79.1,88.8
std,15.5,19.6,14.8,10.2,14.0,14.9,21.5,18.2
min,0.0,0.0,0.0,0.0,0.0,18.2,13.6,0.0
25%,6.2,21.1,8.8,0.0,9.0,87.7,65.8,85.2
50%,12.5,31.4,12.5,7.0,16.7,100.0,86.4,94.9
75%,26.0,49.0,20.2,11.4,27.5,100.0,96.7,100.0
max,71.4,80.0,79.2,50.0,67.7,100.0,100.0,100.0


In [7]:
from scipy.stats import shapiro

print("Teste de Shapiro-Wilk (p < 0.05 = não normal):")
for col in ITENS_INFRA + ["ideb_2023"]:
    stat, p = shapiro(df_merge[col].dropna())
    normal = "normal" if p >= 0.05 else "não normal"
    print(f"  {col:<35} p={p:.4f}  → {normal}")

Teste de Shapiro-Wilk (p < 0.05 = não normal):
  IN_BIBLIOTECA                       p=0.0000  → não normal
  IN_BIBLIOTECA_SALA_LEITURA          p=0.0268  → não normal
  IN_LABORATORIO_INFORMATICA          p=0.0000  → não normal
  IN_LABORATORIO_CIENCIAS             p=0.0000  → não normal
  IN_QUADRA_ESPORTES                  p=0.0004  → não normal
  IN_INTERNET                         p=0.0000  → não normal
  IN_COMPUTADOR                       p=0.0000  → não normal
  IN_BANDA_LARGA                      p=0.0000  → não normal
  ideb_2023                           p=0.0000  → não normal


### Justificativa da escolha do coeficiente de correlação

Foi utilizada a correlação de **Spearman** em vez de Pearson pelos seguintes motivos:

- **Pearson** assume distribuição normal nas variáveis e mede correlação linear
- **Spearman** trabalha com postos (rankings) dos valores, sendo robusto a
  distribuições assimétricas e outliers

O teste de Shapiro-Wilk confirmou que **todas as variáveis analisadas têm
distribuição não normal** (p < 0.05 em todos os casos), incluindo o próprio
IDEB 2023. Isso invalida o uso de Pearson e valida a escolha de Spearman
como o coeficiente mais adequado para esta análise.

In [5]:
from scipy import stats

# Carregar IDEB 2023 EF Anos Iniciais por município
df_ideb = pd.read_parquet(IDEB_SERIES_PARQUET)

ideb_2023 = (
    df_ideb[
        (df_ideb["etapa"] == "EF Anos Iniciais") &
        (df_ideb["ano"] == 2023)
    ]
    [["CO_MUNICIPIO", "NO_MUNICIPIO", "ideb"]]
    .rename(columns={"ideb": "ideb_2023"})
)

# Merge infraestrutura + IDEB
df_merge = df_infra_mun.merge(ideb_2023, on="CO_MUNICIPIO", how="inner")

print(f"Municípios com infraestrutura e IDEB 2023: {len(df_merge)}")

# Calcular correlação de Spearman para cada item
print("\nCorrelação de Spearman com IDEB 2023:")
print(f"{'Item':<35} {'r':>6}  {'p-valor':>8}  {'Significativo':>13}")
print("-" * 68)

resultados = []
for col in ITENS_INFRA:
    sub = df_merge[["ideb_2023", col]].dropna()
    r, p = stats.spearmanr(sub[col], sub["ideb_2023"])
    sig = "✓" if p < 0.05 else "✗"
    resultados.append({"item": col, "r": round(r, 3), "p": round(p, 4), "sig": sig})
    print(f"{col:<35} {r:>6.3f}  {p:>8.4f}  {sig:>13}")

df_corr = pd.DataFrame(resultados).sort_values("r", ascending=False)

Municípios com infraestrutura e IDEB 2023: 100

Correlação de Spearman com IDEB 2023:
Item                                     r   p-valor  Significativo
--------------------------------------------------------------------
IN_BIBLIOTECA                        0.074    0.4650              ✗
IN_BIBLIOTECA_SALA_LEITURA           0.013    0.8996              ✗
IN_LABORATORIO_INFORMATICA           0.086    0.3966              ✗
IN_LABORATORIO_CIENCIAS             -0.048    0.6378              ✗
IN_QUADRA_ESPORTES                   0.019    0.8518              ✗
IN_INTERNET                          0.064    0.5247              ✗
IN_COMPUTADOR                       -0.004    0.9698              ✗
IN_BANDA_LARGA                      -0.022    0.8274              ✗


In [6]:
# Verificar a distribuição do IDEB 2023 no merge
print("IDEB 2023 no dataset de merge:")
print(df_merge["ideb_2023"].describe().round(2))

print(f"\nMunicípios com IDEB 2023 no parquet: {len(ideb_2023)}")
print(f"Municípios no merge: {len(df_merge)}")
print(f"Municípios perdidos no merge: {len(ideb_2023) - len(df_merge)}")

# Ver dispersão de um item vs IDEB para confirmar ausência de padrão
print("\nCorrelação detalhada — IN_BIBLIOTECA_SALA_LEITURA vs IDEB 2023:")
sub = df_merge[["NO_MUNICIPIO", "IN_BIBLIOTECA_SALA_LEITURA", "ideb_2023"]].dropna()
print(sub.sort_values("IN_BIBLIOTECA_SALA_LEITURA", ascending=False).head(10).to_string(index=False))

IDEB 2023 no dataset de merge:
count    100.00
mean       5.91
std        1.20
min        4.20
25%        5.20
50%        5.60
75%        6.10
max        9.80
Name: ideb_2023, dtype: float64

Municípios com IDEB 2023 no parquet: 100
Municípios no merge: 100
Municípios perdidos no merge: 0

Correlação detalhada — IN_BIBLIOTECA_SALA_LEITURA vs IDEB 2023:
   NO_MUNICIPIO  IN_BIBLIOTECA_SALA_LEITURA  ideb_2023
  Feliz Deserto                        80.0        7.0
   Campo Alegre                        76.0        7.3
   Boca da Mata                        72.7        5.9
        Jacuípe                        71.4        5.7
Teotônio Vilela                        70.7        9.0
         Maceió                        70.1        5.3
      Flexeiras                        70.0        5.6
     Paripueira                        66.7        4.9
      Junqueiro                        65.4        8.4
      Arapiraca                        61.9        5.5


In [ ]:
import plotly.express as px

# Scatter: biblioteca_sala_leitura vs IDEB (item com maior r)
fig = px.scatter(
    df_merge,
    x="IN_BIBLIOTECA_SALA_LEITURA",
    y="ideb_2023",
    custom_data=["NO_MUNICIPIO", "IN_BIBLIOTECA_SALA_LEITURA", "ideb_2023"],
    color="ideb_2023",
    color_continuous_scale=["#D85A30", "#F1EFE8", "#1D9E75"],
    labels={
        "IN_BIBLIOTECA_SALA_LEITURA": "% escolas com biblioteca ou sala de leitura",
        "ideb_2023": "IDEB 2023",
    },
    title="Infraestrutura vs IDEB 2023 — Municípios de Alagoas<br>"
          "<sup>Biblioteca/sala de leitura: item com maior correlação (r=0.013, p=0.90)</sup>",
    height=480,
)

fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Biblioteca/sala leitura: %{customdata[1]:.1f}%<br>"
        "IDEB 2023: %{customdata[2]:.2f}<br>"
        "<extra></extra>"
    ),
)

# Linha de tendência
import numpy as np
x = df_merge["IN_BIBLIOTECA_SALA_LEITURA"].dropna()
y = df_merge.loc[x.index, "ideb_2023"]
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
fig.add_scatter(
    x=x_line, y=m * x_line + b,
    mode="lines",
    line=dict(color="#888780", width=1.5, dash="dash"),
    name="Tendência",
    hoverinfo="skip",
)

fig.update_coloraxes(showscale=False)
fig.update_layout(showlegend=False)
fig.show()

**O gráfico mostra a relação entre duas variáveis para os 100 municípios de Alagoas:**

**Eixo X — horizontal**
Percentual de escolas do município que têm biblioteca ou sala de leitura. Um município com percentual 0% não tem nenhuma escola com biblioteca ou sala de leitura. Um município com percentual 80% tem 80% das suas escolas com biblioteca ou sala de leitura.

**Eixo Y — vertical**
IDEB 2023 do município. Quanto mais alto, melhor o desempenho.

**Cada ponto**
É um município. A cor segue o IDEB — pontos vermelhos são municípios com IDEB mais baixo, pontos verdes são os com IDEB mais alto.

**A linha tracejada cinza**
É a linha de tendência calculada por regressão linear. Se houvesse correlação forte entre as duas variáveis, essa linha estaria inclinada — subindo da esquerda para a direita (correlação positiva) ou descendo (correlação negativa). No nosso caso ela está quase completamente horizontal, o que visualmente confirma o coeficiente de correlação r=0.013, ou seja, a relação entre as duas variáveis é praticamente inexistente.

**O que o gráfico revela intuitivamente**
Se você cobrir a linha de tendência e olhar só para os pontos, vai notar que municípios com 0% de cobertura de biblioteca têm IDEBs espalhados do mais baixo ao mais alto — e o mesmo acontece com municípios com 70% ou 80% de cobertura. Os pontos estão distribuídos de forma bastante aleatória verticalmente, independente da posição horizontal. Isso é a ausência de correlação visível a olho nu.

**Por que escolhemos esse item para o gráfico**
Usamos IN_BIBLIOTECA_SALA_LEITURA porque foi o item com o maior coeficiente r entre todos os analisados (r=0.013), ou seja, o que mais se aproximou de ter alguma correlação — mesmo assim praticamente zero. Se a correlação mais forte já é essa, os outros itens seriam ainda menos informativos no gráfico.

### Interpretação

**Resultado principal:** nenhum item de infraestrutura analisado apresentou
correlação estatisticamente significativa com o IDEB 2023 em Alagoas
(todos os p-valores > 0.05).

**Por que isso é relevante:**
A ausência de correlação sugere que a simples presença de infraestrutura
física nas escolas — biblioteca, laboratório, quadra, internet — não é
suficiente para explicar diferenças de desempenho entre os municípios
alagoanos. Municípios com cobertura similar de infraestrutura apresentam
IDEBs muito distintos.

**Exemplos concretos:**
Teotônio Vilela e Maceió têm cobertura semelhante de biblioteca/sala de leitura
(~70%), mas IDEB 9.0 e 5.3 respectivamente. Isso evidencia que outros
fatores — qualidade docente, gestão escolar, contexto socioeconômico podem
ter papel mais determinante que a infraestrutura física.

**Limitação metodológica:**
A análise usa a *presença* do item (0 ou 1 por escola), não a *qualidade*
ou *utilização* da infraestrutura. Uma biblioteca mal equipada ou pouco
utilizada conta igualmente a uma bem estruturada. Dados mais granulares
poderiam revelar correlações que esta análise não captura.